# Fetch data

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
y = y.iloc[:, 0]

# Prepare attributes, pipeline and split sets

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

# NaNN and NaN --> np.nan
X = X.replace(["NaN", "NaNN", "", " "], np.nan)

# Change categorical variables to numeric
cols_to_numeric = ["Cholesterol", "Copper", "Tryglicerides", "Platelets"]
X[cols_to_numeric] = X[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Change Stage to category
X["Stage"] = X["Stage"].astype("category")

# Split data into training and test sets
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

# Cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

# Preprocessor
cat_cols = X_rest.select_dtypes(include=["object", "str", "category"]).columns
num_cols = X_rest.select_dtypes(include=["number"]).columns

# Preprocessor with imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols,),
    ]
)

# Bayes classificator

In [3]:

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier

model = Pipeline([
    ("prep", preprocessor),
    ("dt", DecisionTreeClassifier())
])

param_grid = {
    # DecisionTreeClassifier
    'dt__max_depth': [3, 4, 5, 6],
    'dt__min_samples_split': [2, 5, 10, 20],
    'dt__min_samples_leaf': [1, 2, 4, 6],
    'dt__criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator=model, 
    param_grid=param_grid, 
    cv=cv, 
    scoring="accuracy", 
    n_jobs=-1, # All available cores
)

grid_search.fit(X_rest, y_rest)

print("Best params")
for param_name in sorted(param_grid.keys()):
    print(f"\t{param_name}: {grid_search.best_params_[param_name]}")
print(f"\nBest Accuracy from cv: {grid_search.best_score_:.4f}\n")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Test Set (Zoptymalizowany Model) - Classification Report\n")
print(classification_report(y_test, y_pred_best))


Best params
	dt__criterion: entropy
	dt__max_depth: 3
	dt__min_samples_leaf: 1
	dt__min_samples_split: 2

Best Accuracy from cv: 0.7218

Test Set (Zoptymalizowany Model) - Classification Report

              precision    recall  f1-score   support

           C       0.69      0.85      0.76        47
          CL       0.00      0.00      0.00         5
           D       0.62      0.50      0.55        32

    accuracy                           0.67        84
   macro avg       0.44      0.45      0.44        84
weighted avg       0.62      0.67      0.64        84



s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh